# Day 7 · Lab 1 — Insight Synthesis Agent

## What you'll build

1. Load sample Q4 data (SQL results + extracted requirements)
2. Build an insight-extraction prompt that asks for SURPRISE, not summary
3. Parse structured JSON output (Pydantic + manual json.loads for reliability)
4. Rank insights by surprise_level × confidence
5. Verify traceability — every insight's supporting_data links to source

## Prerequisites

- Track 3.A + Days 5-6 completed
- Same sandbox setup

## Step 1 — Environment

In [ ]:
import os, sys, subprocess, json
from pathlib import Path

for pkg in ["python-dotenv", "langchain-openai"]:
    try: __import__(pkg.replace("-", "_").split("[")[0])
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--user", pkg])

from dotenv import load_dotenv
load_dotenv(Path("~/agentic-lab/.env").expanduser(), override=False)
for k in ("ANTHROPIC_API_KEY","OPENAI_API_KEY","LANGSMITH_API_KEY"):
    if os.environ.get(k) == "": del os.environ[k]

assert os.environ.get("OPENROUTER_API_KEY"), "OPENROUTER_API_KEY missing"
os.environ["OPENAI_API_KEY"] = os.environ["OPENROUTER_API_KEY"]
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"
print("✓ Ready")

## Step 2 — Sample Q4 data + requirements

In [ ]:
# Simulated Q4 data — in production these come from Day 5 SQL results
q4_data = {
    "revenue_by_region": {"NA": 8_200_000, "EU": 3_100_000, "APAC": 1_100_000},
    "revenue_prior_year_q4": {"NA": 7_800_000, "EU": 2_400_000, "APAC": 380_000},
    "new_customers": {"NA": 42, "EU": 28, "APAC": 71},
    "churn_rate_pct": {"NA": 2.1, "EU": 3.8, "APAC": 1.2},
    "csat_score": {"NA": 8.4, "EU": 7.9, "APAC": 8.9},
    "support_tickets": {"NA": 3_400, "EU": 2_800, "APAC": 1_100},
}

# Simulated requirements from Day 6 (previously discussed initiatives)
requirements_context = [
    {"id": "REQ-01", "type": "business", "statement": "APAC expansion target: 40% YoY revenue growth"},
    {"id": "REQ-02", "type": "business", "statement": "Reduce EU churn below 3.0% by end of year"},
    {"id": "REQ-03", "type": "functional", "statement": "Self-service portal to reduce ticket volume by 40%"},
]

print("Q4 data loaded")
print(f"  Regions: {list(q4_data['revenue_by_region'].keys())}")
print(f"  Metrics: {list(q4_data.keys())}")
print(f"  Requirements: {len(requirements_context)}")

## Step 3 — Insight extraction prompt

The Day 7 core lesson: prompt for SURPRISE, not summary.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="anthropic/claude-sonnet-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0,
)


INSIGHT_PROMPT_TEMPLATE = '''You are an experienced BA writing insights for a CFO's Q4 review.

Data:
{data}

Prior stated goals:
{goals}

Your task: extract the TOP 5 insights that would MOST SURPRISE a CFO who hasn't seen this data yet.

For each insight, output as JSON in the following schema:
{{
  "insights": [
    {{
      "claim": "concise insight statement (e.g., 'APAC revenue tripled YoY, exceeding target')",
      "supporting_data": ["data point 1", "data point 2"],
      "surprise_level": "low" | "medium" | "high",
      "recommended_action": "specific action a CFO could take",
      "confidence": 0.0-1.0,
      "linked_requirement_id": "REQ-XX or null"
    }}
  ]
}}

Rules:
- Every claim MUST cite specific numbers from the data
- surprise_level: 'high' if this would change the CFO's priorities; 'medium' if useful; 'low' if expected
- No adjectives without data: 'strong growth' → '18% growth'
- Skip anything that would not change decisions

Return ONLY the JSON. No markdown fences, no prose.'''


prompt = INSIGHT_PROMPT_TEMPLATE.format(
    data=json.dumps(q4_data, indent=2),
    goals=json.dumps(requirements_context, indent=2),
)

print("Prompt ready. Length:", len(prompt), "chars")

## Step 4 — Extract insights (manual JSON parse — reliable on OpenRouter)

In [ ]:
raw = llm.invoke(prompt).content.strip()

# Strip markdown fences if the model added them anyway
if raw.startswith("```"):
    raw = raw.strip("`").split("\n", 1)[-1].rsplit("```", 1)[0].strip()
    if raw.startswith("json"):
        raw = raw[4:].strip()

try:
    data = json.loads(raw)
    insights = data["insights"]
    print(f"✓ Parsed {len(insights)} insights")
except json.JSONDecodeError as e:
    print(f"✗ JSON parse failed: {e}")
    print(f"Raw output:\n{raw[:500]}")
    raise

## Step 5 — Rank by surprise × confidence

In [ ]:
LEVEL_WEIGHT = {"high": 3, "medium": 2, "low": 1}


def score(insight):
    return LEVEL_WEIGHT.get(insight["surprise_level"], 1) * insight["confidence"]


ranked = sorted(insights, key=score, reverse=True)

print("═" * 60)
print("TOP 5 INSIGHTS (by surprise × confidence)")
print("═" * 60)
for i, ins in enumerate(ranked[:5], 1):
    print(f"\n#{i} [{ins['surprise_level']} surprise, conf={ins['confidence']:.2f}]")
    print(f"  Claim: {ins['claim']}")
    print(f"  Action: {ins['recommended_action']}")
    if ins.get('linked_requirement_id'):
        print(f"  Linked req: {ins['linked_requirement_id']}")

## Step 6 — Verify traceability

Every insight's supporting_data should reference numbers actually in q4_data.

In [ ]:
import re


def flatten_data_numbers(d):
    """Get every numeric value in the source data as strings."""
    nums = set()
    def walk(x):
        if isinstance(x, dict):
            for v in x.values(): walk(v)
        elif isinstance(x, (list, tuple)):
            for v in x: walk(v)
        elif isinstance(x, (int, float)):
            nums.add(str(x))
            # Also add formatted versions
            if isinstance(x, int) and x >= 1000:
                nums.add(f"{x:,}")
    walk(d)
    return nums


source_numbers = flatten_data_numbers(q4_data)
print(f"Source data has {len(source_numbers)} distinct numeric values")

# Check each insight's claim: does it cite a number from source?
verified = 0
for ins in ranked[:5]:
    numbers_in_claim = set(re.findall(r'\d[\d,]*\.?\d*', ins['claim']))
    # Any overlap with source numbers OR a percentage that's plausible
    if numbers_in_claim & source_numbers:
        verified += 1
    elif any('%' in ins['claim'] for _ in [1]):  # % might be derived
        verified += 0.5

print(f"Traceability: {verified}/{len(ranked[:5])} insights cite verifiable numbers or derived percentages")

## Step 7 — Save for Lab 2

In [ ]:
out_path = Path("/tmp/day7_insights.json")
out_path.write_text(json.dumps({
    "insights": ranked[:5],
    "source_data": q4_data,
    "requirements": requirements_context,
}, indent=2))
print(f"✓ Saved to {out_path}")
print(f"  Top 5 insights available for Lab 2")

## What you learned

1. **Prompt for surprise** — 'what would surprise the CFO?' beats 'summarize the data'
2. **Manual JSON parsing** works reliably on OpenRouter (structured_output can misbehave)
3. **Surprise × confidence** as a ranking function separates signal from noise
4. **Traceability** at the report level: numbers in claims should trace to source data
5. **Insight is not summary** — recommended_action makes it actionable

## Next

Open `lab2_report_charts.ipynb` for report generation with charts + consistency verification.